# 02 · Ingest the bronze layer

Two heterogeneous sources land in Iceberg:

- **Postgres** (`oneshop`) → `demo.bronze.{users,items,purchases}` over **JDBC**
- **SeaweedFS** `pageviews` bucket (raw JSON) → `demo.bronze.pageviews` over **s3a://**

> Make sure data exists first: run the loadgen (`make pipeline` runs it for you,
> or `kubectl apply -f k8s/60-loadgen.yaml`).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("02-ingest-bronze").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

## Postgres → bronze (JDBC)

In [ ]:
JDBC = {
    "url": "jdbc:postgresql://postgres:5432/oneshop",
    "user": "etluser",
    "password": "etlpassword",
    "driver": "org.postgresql.Driver",
}

def read_pg(table):
    return spark.read.format("jdbc").options(**JDBC).option("dbtable", table).load()

users = read_pg("users")
items = read_pg("items")
purchases = read_pg("purchases")
print("users:", users.count(), "| items:", items.count(), "| purchases:", purchases.count())
users.show(3)

Cast to the bronze schema and overwrite (full reload).

In [ ]:
from pyspark.sql.functions import col

def overwrite(df, table):
    df.write.format("iceberg").mode("overwrite").save(table)
    print(table, "->", df.count(), "rows")

overwrite(users.select(
    col("id").cast("long"), col("first_name").cast("string"), col("last_name").cast("string"),
    col("email").cast("string"), col("created_at").cast("timestamp"), col("updated_at").cast("timestamp"),
), "demo.bronze.users")

overwrite(items.select(
    col("id").cast("long"), col("name").cast("string"), col("category").cast("string"),
    col("price").cast("decimal(7,2)"), col("inventory").cast("int"),
    col("created_at").cast("timestamp"), col("updated_at").cast("timestamp"),
), "demo.bronze.items")

overwrite(purchases.select(
    col("id").cast("long"), col("user_id").cast("long"), col("item_id").cast("long"),
    col("quantity").cast("int"), col("purchase_price").cast("decimal(12,2)"),
    col("created_at").cast("timestamp"), col("updated_at").cast("timestamp"),
), "demo.bronze.purchases")

## Pageview JSON → bronze (s3a://)

In [ ]:
raw = spark.read.json("s3a://pageviews/")
print("raw pageview events:", raw.count())
raw.show(3, truncate=False)

In [ ]:
pageviews = raw.select(
    col("user_id").cast("long"),
    col("url").cast("string"),
    col("channel").cast("string"),
    col("received_at").cast("timestamp"),   # epoch seconds -> timestamp
)
overwrite(pageviews, "demo.bronze.pageviews")

## Peek at the bronze layer

In [ ]:
%%sql
SELECT 'users' t, count(*) n FROM demo.bronze.users
UNION ALL SELECT 'items', count(*) FROM demo.bronze.items
UNION ALL SELECT 'purchases', count(*) FROM demo.bronze.purchases
UNION ALL SELECT 'pageviews', count(*) FROM demo.bronze.pageviews